# 🧴 Cosmetic Market Intelligence — ML Training Notebook
**End-to-end training, EDA, and evaluation for:**
1. Ingredient Similarity (TF-IDF + Cosine)
2. Harmful Ingredient Detector (Rule-based)
3. Skin-Type Classifier (TF-IDF + LogisticRegression)
4. Sentiment Analysis (VADER / Supervised)
5. Demand Forecasting (ARIMA + RandomForest)

---
> **Run all cells sequentially.** Models are saved to `backend/models/`.


## 0 · Imports & Setup

In [ ]:
import os, sys, warnings, json, pickle
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from collections import Counter
warnings.filterwarnings('ignore')

# ── Plot style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})
sns.set_palette('viridis')

# ── Paths ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
# Adjust if running from project root vs backend/
if (NOTEBOOK_DIR / 'data').exists():
    BASE_DIR = NOTEBOOK_DIR
else:
    BASE_DIR = NOTEBOOK_DIR.parent / 'backend'

DATA_DIR  = BASE_DIR / 'data' / 'cleaned'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Base  : {BASE_DIR}')
print(f'Data  : {DATA_DIR}')
print(f'Models: {MODELS_DIR}')


## 1 · Load Datasets

In [ ]:
df_master  = pd.read_csv(DATA_DIR / 'master_cleaned.csv',   low_memory=False)
df_sales   = pd.read_csv(DATA_DIR / 'master_sales_cleaned.csv', low_memory=False)
df_reviews = pd.read_csv(DATA_DIR / 'master_reviews_cleaned.csv', low_memory=False)

print(f'master_cleaned   : {df_master.shape}')
print(f'master_sales     : {df_sales.shape}')
print(f'master_reviews   : {df_reviews.shape}')


In [ ]:
df_master.head(3)


## 2 · Exploratory Data Analysis (EDA)

### 2.1 · Missing Values Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (df, name) in zip(axes, [
    (df_master,  'master_cleaned'),
    (df_sales,   'master_sales'),
    (df_reviews, 'master_reviews'),
]):
    missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False).head(15)
    missing_pct.plot(kind='barh', ax=ax, color='#6c5ce7')
    ax.set_title(f'{name} — Missing %')
    ax.set_xlabel('% Missing')
    ax.axvline(50, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_missing_values.png', bbox_inches='tight')
plt.show()


### 2.2 · Skin Type Distribution

In [ ]:
if 'skin_suitability' in df_master.columns:
    skin_counts = df_master['skin_suitability'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.Set2(range(len(skin_counts)))
    skin_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title('Skin Type Suitability Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Skin Type')
    ax.set_ylabel('Number of Products')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'eda_skin_distribution.png', bbox_inches='tight')
    plt.show()
else:
    print('skin_suitability column not found')


### 2.3 · Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'average_rating' in df_master.columns:
    avg_rating = pd.to_numeric(df_master['average_rating'], errors='coerce').dropna()
    axes[0].hist(avg_rating, bins=30, color='#00b894', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Average Rating Distribution (Products)')
    axes[0].set_xlabel('Rating')
    axes[0].axvline(avg_rating.mean(), color='red', linestyle='--',
                    label=f'Mean: {avg_rating.mean():.2f}')
    axes[0].legend()

if 'rating_value' in df_reviews.columns:
    rev_rating = pd.to_numeric(df_reviews['rating_value'], errors='coerce').dropna()
    axes[1].hist(rev_rating, bins=20, color='#e17055', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Review Rating Distribution')
    axes[1].set_xlabel('Rating')
    axes[1].axvline(rev_rating.mean(), color='navy', linestyle='--',
                    label=f'Mean: {rev_rating.mean():.2f}')
    axes[1].legend()

plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_rating_distributions.png', bbox_inches='tight')
plt.show()


### 2.4 · Top Brands by Product Count

In [ ]:
if 'brand' in df_master.columns:
    top_brands = df_master['brand'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(top_brands.index[::-1], top_brands.values[::-1],
                   color=plt.cm.plasma(np.linspace(0.2, 0.8, len(top_brands))))
    ax.set_title('Top 15 Brands by Product Count', fontweight='bold')
    ax.set_xlabel('Number of Products')
    for bar, val in zip(bars, top_brands.values[::-1]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'eda_top_brands.png', bbox_inches='tight')
    plt.show()


### 2.5 · Safety Score Distribution

In [ ]:
if 'safety_score' in df_master.columns:
    scores = pd.to_numeric(df_master['safety_score'], errors='coerce').dropna()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Histogram
    axes[0].hist(scores, bins=30, color='#0984e3', edgecolor='white')
    axes[0].set_title('Safety Score Distribution')
    axes[0].set_xlabel('Safety Score (0-100)')
    axes[0].axvline(80, color='green', linestyle='--', label='Safe threshold (80)')
    axes[0].axvline(50, color='red',   linestyle='--', label='Unsafe threshold (50)')
    axes[0].legend()
    # Pie
    cats = pd.cut(scores, bins=[0, 50, 80, 100], labels=['Unsafe', 'Moderate', 'Safe'])
    cat_counts = cats.value_counts()
    axes[1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
                colors=['#d63031', '#fdcb6e', '#00b894'],
                startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    axes[1].set_title('Safety Category Split')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'eda_safety_scores.png', bbox_inches='tight')
    plt.show()
    print(f'Mean safety score: {scores.mean():.2f}')


### 2.6 · Sales Time Series Overview

In [ ]:
date_col = None
qty_col  = None
for c in ['start_date', 'order_date', 'date']:
    if c in df_sales.columns: date_col = c; break
for c in ['units_sold', 'sales', 'revenues']:
    if c in df_sales.columns: qty_col = c; break

if date_col and qty_col:
    ts_df = df_sales[[date_col, qty_col]].copy()
    ts_df[date_col] = pd.to_datetime(ts_df[date_col], errors='coerce')
    ts_df[qty_col]  = pd.to_numeric(ts_df[qty_col], errors='coerce')
    ts_df = ts_df.dropna().set_index(date_col).sort_index()
    weekly = ts_df[qty_col].resample('W').sum()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(weekly.index, weekly.values, color='#6c5ce7', linewidth=1.5)
    ax.fill_between(weekly.index, weekly.values, alpha=0.15, color='#6c5ce7')
    ax.set_title('Weekly Total Sales (All Products)', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Units Sold')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'eda_weekly_sales.png', bbox_inches='tight')
    plt.show()
    print(f'Date range: {weekly.index.min().date()} → {weekly.index.max().date()}')
else:
    print('Could not identify date/qty columns for time series plot')


### 2.7 · Word Cloud — Review Text

In [ ]:
try:
    from wordcloud import WordCloud
    if 'review_text' in df_reviews.columns:
        text_sample = ' '.join(df_reviews['review_text'].dropna().astype(str).head(3000).tolist())
        wc = WordCloud(
            width=1400, height=700,
            background_color='white',
            max_words=200,
            colormap='viridis',
            prefer_horizontal=0.85,
        ).generate(text_sample)
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Word Cloud — Review Text', fontsize=16, fontweight='bold', pad=10)
        plt.tight_layout()
        plt.savefig(MODELS_DIR / 'eda_wordcloud_reviews.png', bbox_inches='tight')
        plt.show()
except ImportError:
    print('wordcloud not installed: pip install wordcloud')


### 2.8 · Word Cloud — Ingredients

In [ ]:
try:
    from wordcloud import WordCloud
    ing_col = 'ingredients' if 'ingredients' in df_reviews.columns else 'ingridients'
    if ing_col in df_reviews.columns:
        ing_text = ' '.join(df_reviews[ing_col].dropna().astype(str).head(2000).tolist())
        wc2 = WordCloud(
            width=1400, height=700,
            background_color='#1e1e2e',
            max_words=150,
            colormap='cool',
        ).generate(ing_text)
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(wc2, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Word Cloud — Ingredients', fontsize=16, fontweight='bold', pad=10)
        plt.tight_layout()
        plt.savefig(MODELS_DIR / 'eda_wordcloud_ingredients.png', bbox_inches='tight')
        plt.show()
except ImportError:
    pass


---
## 3 · Model 1 — Ingredient Similarity (TF-IDF + Cosine)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ing_col = 'ingredients_clean' if 'ingredients_clean' in df_master.columns else 'ingredients'
subset_sim = df_master[['product_id', 'product_name', ing_col]].dropna(subset=['product_id']).copy()
subset_sim['ing_text'] = subset_sim[ing_col].fillna('').astype(str).str.lower().str.strip()
subset_sim = subset_sim[subset_sim['ing_text'].str.len() > 2]
print(f'Products with ingredient text: {len(subset_sim):,}')

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(subset_sim['ing_text'])
print(f'TF-IDF matrix: {tfidf_matrix.shape}')

print('Computing cosine similarity...')
sim_matrix = cosine_similarity(tfidf_matrix, dense_output=False)
print(f'Similarity matrix: {sim_matrix.shape}')

product_ids   = list(subset_sim['product_id'].values)
product_names = list(subset_sim['product_name'].values)


In [ ]:
# Visualise top-10 TF-IDF features
feature_names = tfidf_vectorizer.get_feature_names_out()
sample_vector = tfidf_matrix[0].toarray()[0]
top_idx = sample_vector.argsort()[::-1][:15]
top_feats = [(feature_names[i], sample_vector[i]) for i in top_idx if sample_vector[i] > 0]

fig, ax = plt.subplots(figsize=(12, 5))
if top_feats:
    names, scores = zip(*top_feats)
    ax.barh(list(names)[::-1], list(scores)[::-1], color='#a29bfe')
    ax.set_title(f"Top TF-IDF Features — '{product_names[0]}'", fontweight='bold')
    ax.set_xlabel('TF-IDF Weight')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'model1_tfidf_features.png', bbox_inches='tight')
plt.show()


In [ ]:
# Show top-5 similar to first product
idx = 0
sims = sim_matrix[idx].toarray().flatten()
top5 = np.argsort(sims)[::-1][1:6]
print(f"Top-5 similar to '{product_names[idx]}':")
for i, t in enumerate(top5, 1):
    print(f'  {i}. {product_names[t]:<50s}  similarity={sims[t]:.4f}')

# Similarity heatmap (top 20 products)
n = min(20, len(product_ids))
sub_sim = sim_matrix[:n, :n].toarray()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sub_sim, ax=ax, cmap='YlOrRd', square=True,
            xticklabels=product_names[:n], yticklabels=product_names[:n],
            linewidths=0.3)
ax.set_title('Ingredient Cosine Similarity Heatmap (Top 20 Products)', fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'model1_similarity_heatmap.png', bbox_inches='tight')
plt.show()


In [ ]:
# Save artefacts
with open(MODELS_DIR / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(MODELS_DIR / 'cosine_similarity_matrix.pkl', 'wb') as f:
    pickle.dump(sim_matrix, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(MODELS_DIR / 'product_id_index.pkl', 'wb') as f:
    pickle.dump({'product_ids': product_ids, 'product_names': product_names}, f,
                protocol=pickle.HIGHEST_PROTOCOL)
print('✔ Similarity models saved')


---
## 4 · Model 2 — Harmful Ingredient Detector

In [ ]:
HARMFUL_KEYWORDS = {
    'paraben':               {'name': 'Parabens',             'reason': 'Endocrine disruptor',          'severity': 7},
    'methylparaben':         {'name': 'Methylparaben',         'reason': 'Endocrine disruptor',          'severity': 7},
    'propylparaben':         {'name': 'Propylparaben',         'reason': 'Endocrine disruptor',          'severity': 7},
    'sodium lauryl sulfate': {'name': 'SLS',                   'reason': 'Skin irritant',                'severity': 5},
    'sodium laureth sulfate':{'name': 'SLES',                  'reason': 'Possible carcinogen contamination', 'severity': 6},
    'sls':                   {'name': 'SLS abbreviation',      'reason': 'Skin irritant',                'severity': 5},
    'oxybenzone':            {'name': 'Oxybenzone',            'reason': 'Endocrine disruptor',          'severity': 8},
    'phthalate':             {'name': 'Phthalates',            'reason': 'Reproductive toxicant',        'severity': 8},
    'formaldehyde':          {'name': 'Formaldehyde',          'reason': 'Known carcinogen (IARC G1)',   'severity': 10},
    'quaternium-15':         {'name': 'Quaternium-15',         'reason': 'Formaldehyde releaser',        'severity': 9},
    'dmdm hydantoin':        {'name': 'DMDM Hydantoin',        'reason': 'Formaldehyde releaser',        'severity': 9},
    'peg':                   {'name': 'PEGs',                  'reason': '1,4-dioxane contamination',   'severity': 5},
    'petroleum':             {'name': 'Petroleum',             'reason': 'PAH contamination risk',       'severity': 5},
    'phenoxyethanol':        {'name': 'Phenoxyethanol',        'reason': 'Potential skin sensitiser',    'severity': 4},
    'triclosan':             {'name': 'Triclosan',             'reason': 'Antibiotic resistance',        'severity': 8},
    'coal tar':              {'name': 'Coal Tar',              'reason': 'Known carcinogen',             'severity': 9},
    'hydroquinone':          {'name': 'Hydroquinone',          'reason': 'Restricted in EU',             'severity': 7},
}

def compute_safety_score(text):
    if not text or str(text).strip().lower() in ('', 'nan'): return 100.0, []
    text = str(text).lower()
    found, total_sev = [], 0
    for kw, info in HARMFUL_KEYWORDS.items():
        if kw in text:
            found.append(info)
            total_sev += info['severity']
    score = max(0.0, 100.0 - min(total_sev * 4, 100))
    return round(score, 1), found


In [ ]:
# Apply to dataset and visualise
ing_col_r = 'ingredients' if 'ingredients' in df_reviews.columns else 'ingridients'
sample_r   = df_reviews[[ing_col_r]].dropna().head(5000).copy()
results    = sample_r[ing_col_r].apply(lambda x: pd.Series(compute_safety_score(str(x)),
                                                            index=['safety_score', 'harmful_list']))
sample_r   = pd.concat([sample_r, results], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sample_r['safety_score'], bins=30, color='#00cec9', edgecolor='white')
axes[0].axvline(80, color='green', linestyle='--', label='Safe ≥ 80')
axes[0].axvline(50, color='red',   linestyle='--', label='Unsafe < 50')
axes[0].set_title('Safety Score Distribution (Reviews Dataset)')
axes[0].set_xlabel('Safety Score')
axes[0].legend()

cats = pd.cut(sample_r['safety_score'], bins=[0,50,80,100], labels=['Unsafe','Moderate','Safe'])
cat_vc = cats.value_counts()
axes[1].pie(cat_vc, labels=cat_vc.index, autopct='%1.1f%%',
            colors=['#d63031','#fdcb6e','#00b894'],
            startangle=140, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Safety Category Split')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'model2_safety_distribution.png', bbox_inches='tight')
plt.show()

all_found = [i['name'] for lst in sample_r['harmful_list'] if isinstance(lst, list) for i in lst]
top_harm  = Counter(all_found).most_common(12)
print('\nTop harmful ingredients found:')
for name, cnt in top_harm:
    print(f'  {cnt:5,}×  {name}')


In [ ]:
if top_harm:
    names, counts = zip(*top_harm)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(list(names)[::-1], list(counts)[::-1], color='#e17055')
    ax.set_title('Most Common Harmful Ingredients Detected', fontweight='bold')
    ax.set_xlabel('Occurrences')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'model2_top_harmful.png', bbox_inches='tight')
    plt.show()

detector_payload = {'harmful_keywords': HARMFUL_KEYWORDS, 'scoring_fn': 'compute_safety_score',
                    'stats': {'n_products': len(sample_r), 'mean_safety': sample_r['safety_score'].mean()}}
with open(MODELS_DIR / 'harmful_detector.pkl', 'wb') as f:
    pickle.dump(detector_payload, f, protocol=pickle.HIGHEST_PROTOCOL)
print('✔ harmful_detector.pkl saved')


---
## 5 · Model 3 — Skin-Type Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              ConfusionMatrixDisplay, confusion_matrix)

skin_labels = df_master[['product_id','skin_suitability']].dropna().drop_duplicates(subset='product_id')
review_sub  = df_reviews[['product_id','review_text']].dropna(subset=['review_text'])
merged = review_sub.merge(skin_labels, on='product_id', how='inner')
merged = merged.dropna(subset=['skin_suitability','review_text'])
merged['skin_suitability'] = merged['skin_suitability'].str.strip().str.lower()
cc = merged['skin_suitability'].value_counts()
valid = cc[cc >= 5].index
merged = merged[merged['skin_suitability'].isin(valid)]
print(f'Training samples: {len(merged):,}')
print(f'Classes: {sorted(merged["skin_suitability"].unique())}')


In [ ]:
if len(merged) >= 20:
    X = merged['review_text'].fillna('').astype(str).str.lower().str.strip().values
    y = merged['skin_suitability'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    skin_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
        ('clf',   LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', solver='lbfgs')),
    ])
    skin_pipeline.fit(X_train, y_train)
    y_pred = skin_pipeline.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1w  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1m  = f1_score(y_test, y_pred, average='macro', zero_division=0)
    print(f'Accuracy (test) : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'F1 (weighted)   : {f1w:.4f}')
    print(f'F1 (macro)      : {f1m:.4f}')
    print()
    print(classification_report(y_test, y_pred, zero_division=0))
else:
    print('Not enough data — saving untrained pipeline')
    skin_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000)),
        ('clf',   LogisticRegression(max_iter=500)),
    ])
    y_test = y_pred = []


In [ ]:
# Confusion Matrix
if len(y_test) > 0 and len(set(y_test)) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    cm = confusion_matrix(y_test, y_pred, labels=sorted(set(y_test)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(set(y_test)))
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title('Skin-Type Classifier — Confusion Matrix', fontweight='bold', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'model3_skin_confusion_matrix.png', bbox_inches='tight')
    plt.show()

with open(MODELS_DIR / 'skin_model.pkl', 'wb') as f:
    pickle.dump(skin_pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)
print('✔ skin_model.pkl saved')


---
## 6 · Model 4 — Sentiment Analysis

In [ ]:
# Check for explicit sentiment labels
has_labels = False
label_col  = None
for c in ['sentiment', 'label', 'review_sentiment']:
    if c in df_reviews.columns and df_reviews[c].nunique() >= 2:
        has_labels, label_col = True, c
        break

# Infer from rating if no explicit label
if not has_labels and 'rating_value' in df_reviews.columns:
    df_reviews['sentiment_inferred'] = pd.to_numeric(df_reviews['rating_value'], errors='coerce').apply(
        lambda r: 'positive' if r >= 4 else ('negative' if r <= 2 else 'neutral')
    )
    label_col  = 'sentiment_inferred'
    has_labels = True
    print(f'Inferred sentiment from rating — distribution:')
    print(df_reviews['sentiment_inferred'].value_counts())
else:
    print('Using VADER rule-based approach')


In [ ]:
if has_labels:
    rev_sub = df_reviews[['review_text', label_col]].dropna()
    rev_sub = rev_sub[rev_sub['review_text'].str.strip().str.len() > 5]
    cc2 = rev_sub[label_col].value_counts()
    valid2 = cc2[cc2 >= 5].index
    rev_sub = rev_sub[rev_sub[label_col].isin(valid2)]
    print(f'Samples: {len(rev_sub):,}  Classes: {list(rev_sub[label_col].unique())}')

    Xs = rev_sub['review_text'].fillna('').astype(str).str.lower().values
    ys = rev_sub[label_col].astype(str).values
    Xs_train, Xs_test, ys_train, ys_test = train_test_split(
        Xs, ys, test_size=0.2, random_state=42, stratify=ys)

    sent_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=8000, ngram_range=(1,2), sublinear_tf=True)),
        ('clf',   LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')),
    ])
    sent_pipeline.fit(Xs_train, ys_train)
    ys_pred = sent_pipeline.predict(Xs_test)

    s_acc = accuracy_score(ys_test, ys_pred)
    s_f1  = f1_score(ys_test, ys_pred, average='weighted', zero_division=0)
    print(f'Accuracy   : {s_acc:.4f}')
    print(f'F1 Weighted: {s_f1:.4f}')
    print(classification_report(ys_test, ys_pred, zero_division=0))
    sentiment_payload = {'mode': 'classifier', 'pipeline': sent_pipeline}
else:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader = SentimentIntensityAnalyzer()
    sample_texts = df_reviews['review_text'].dropna().head(3000)
    vader_results = sample_texts.apply(lambda x: ('positive' if vader.polarity_scores(str(x))['compound'] >= 0.05
                                                  else ('negative' if vader.polarity_scores(str(x))['compound'] <= -0.05
                                                  else 'neutral')))
    dist = vader_results.value_counts(normalize=True).round(3).to_dict()
    print(f'VADER distribution: {dist}')
    sentiment_payload = {'mode': 'vader', 'analyzer': vader, 'distribution': dist}
    ys_test = ys_pred = vader_results.values


In [ ]:
# Sentiment distribution bar chart
if has_labels and len(ys_pred) > 0:
    pred_dist = pd.Series(ys_pred).value_counts()
    true_dist = pd.Series(ys_test).value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    true_dist.plot(kind='bar', ax=axes[0], color=['#00b894','#fdcb6e','#d63031'][:len(true_dist)], edgecolor='white')
    axes[0].set_title('True Sentiment Distribution (Test Set)');
    axes[0].set_xlabel('Sentiment'); axes[0].set_ylabel('Count')
    pred_dist.plot(kind='bar', ax=axes[1], color=['#6c5ce7','#a29bfe','#fd79a8'][:len(pred_dist)], edgecolor='white')
    axes[1].set_title('Predicted Sentiment Distribution (Test Set)')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'model4_sentiment_distribution.png', bbox_inches='tight')
    plt.show()

    if len(set(ys_test)) > 1:
        fig2, ax2 = plt.subplots(figsize=(8, 6))
        cm2 = confusion_matrix(ys_test, ys_pred, labels=sorted(set(ys_test)))
        disp2 = ConfusionMatrixDisplay(cm2, display_labels=sorted(set(ys_test)))
        disp2.plot(ax=ax2, cmap='Purples', colorbar=True)
        ax2.set_title('Sentiment — Confusion Matrix', fontweight='bold')
        plt.tight_layout()
        plt.savefig(MODELS_DIR / 'model4_sentiment_confusion.png', bbox_inches='tight')
        plt.show()

with open(MODELS_DIR / 'sentiment_model.pkl', 'wb') as f:
    pickle.dump(sentiment_payload, f, protocol=pickle.HIGHEST_PROTOCOL)
print('✔ sentiment_model.pkl saved')


---
## 7 · Model 5 — Demand Forecasting

### 7.1 · Build Weekly Time Series

In [ ]:
date_col, qty_col, pid_col = None, None, None
for c in ['start_date','order_date','date']: 
    if c in df_sales.columns: date_col = c; break
for c in ['units_sold','sales','revenues']: 
    if c in df_sales.columns: qty_col  = c; break
if 'product_id' in df_sales.columns: pid_col = 'product_id'

print(f'Date col: {date_col}  |  Qty col: {qty_col}  |  PID col: {pid_col}')

if date_col and qty_col:
    ts_df = df_sales[[date_col, pid_col or qty_col, qty_col]].copy() if pid_col \
            else df_sales[[date_col, qty_col]].copy()
    ts_df[date_col] = pd.to_datetime(ts_df[date_col], errors='coerce')
    ts_df[qty_col]  = pd.to_numeric(ts_df[qty_col],  errors='coerce')
    ts_df = ts_df.dropna()

    if pid_col:
        ts_wide = (ts_df.groupby([pd.Grouper(key=date_col, freq='W'), pid_col])[qty_col]
                       .sum().unstack(fill_value=0).sort_index())
    else:
        ts_wide = ts_df.set_index(date_col)[qty_col].resample('W').sum().to_frame().sort_index()

    print(f'Time series shape: {ts_wide.shape}')
    print(f'Date range: {ts_wide.index.min()} → {ts_wide.index.max()}')
    ts_wide.head()


### 7.2 · ARIMA

In [ ]:
try:
    from statsmodels.tsa.arima.model import ARIMA
    from sklearn.metrics import mean_squared_error

    top_n = 5
    if ts_wide.shape[1] > 1:
        top_products = ts_wide.sum().nlargest(top_n).index.tolist()
    else:
        top_products = ts_wide.columns.tolist()

    arima_models = {}
    arima_rmse   = {}

    for pid in top_products:
        try:
            series = ts_wide[pid].dropna()
            if len(series) < 12: continue
            split  = max(4, int(len(series) * 0.8))
            train_, test_ = series.iloc[:split], series.iloc[split:]
            model = ARIMA(train_, order=(1,1,1)).fit()
            if len(test_) > 0:
                fc    = model.forecast(steps=len(test_))
                rmse  = float(np.sqrt(mean_squared_error(test_, fc)))
                arima_rmse[str(pid)] = rmse
                print(f'  {str(pid)[:20]:20s}  RMSE={rmse:.2f}')
            arima_models[str(pid)] = model
        except Exception as e:
            print(f'  {pid}: ARIMA failed — {e}')

    with open(MODELS_DIR / 'arima_model.pkl', 'wb') as f:
        pickle.dump({'models': arima_models, 'top_products': [str(p) for p in top_products]}, f,
                    protocol=pickle.HIGHEST_PROTOCOL)
    print('\n✔ arima_model.pkl saved')
except ImportError:
    print('statsmodels not installed: pip install statsmodels')
    arima_rmse = {}


In [ ]:
# ARIMA forecast plots
if arima_models and ts_wide.shape[1] >= 1:
    pid_to_plot = list(arima_models.keys())[:3]
    fig, axes = plt.subplots(len(pid_to_plot), 1, figsize=(14, 4 * len(pid_to_plot)), squeeze=False)
    for ax, pid in zip(axes.flatten(), pid_to_plot):
        series = ts_wide[pid].dropna()
        split  = max(4, int(len(series) * 0.8))
        train_, test_ = series.iloc[:split], series.iloc[split:]
        model  = arima_models[pid]
        n_fc   = max(len(test_), 8)
        fc     = model.forecast(steps=n_fc)
        fc_idx = pd.date_range(start=series.index[-1], periods=n_fc+1, freq='W')[1:]

        ax.plot(train_.index, train_.values, color='#0984e3', label='Train', linewidth=1.5)
        ax.plot(test_.index,  test_.values,  color='#00b894', label='Test',  linewidth=1.5)
        ax.plot(fc_idx,       fc.values,     color='#e17055', linestyle='--', label='ARIMA Forecast', linewidth=2)
        ax.set_title(f'ARIMA Forecast — Product: {pid}', fontweight='bold')
        ax.set_ylabel('Units Sold')
        ax.legend()
        if pid in arima_rmse:
            ax.text(0.01, 0.95, f'RMSE: {arima_rmse[pid]:.2f}',
                    transform=ax.transAxes, fontsize=9, va='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'model5_arima_forecasts.png', bbox_inches='tight')
    plt.show()


### 7.3 · Random Forest Demand Forecasting

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def engineer_features(ts_series):
    df = pd.DataFrame({'y': ts_series})
    df['month']          = df.index.month
    df['week_of_year']   = df.index.isocalendar().week.astype(int)
    df['quarter']        = df.index.quarter
    df['season']         = (df['quarter'] - 1) % 4 + 1
    df['is_holiday']     = (df['quarter'] == 4).astype(int)
    df['lag_1']          = df['y'].shift(1)
    df['lag_2']          = df['y'].shift(2)
    df['lag_4']          = df['y'].shift(4)
    df['lag_8']          = df['y'].shift(8)
    df['rolling_mean_4'] = df['y'].shift(1).rolling(4).mean()
    return df.dropna()

frames = []
for col in ts_wide.columns:
    feats = engineer_features(ts_wide[col])
    frames.append(feats)

combined = pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
fcols    = [c for c in combined.columns if c != 'y']
X_rf, y_rf = combined[fcols].values, combined['y'].values

split_rf = int(len(combined) * 0.8)
Xr_train, Xr_test = X_rf[:split_rf], X_rf[split_rf:]
yr_train, yr_test = y_rf[:split_rf], y_rf[split_rf:]

print(f'Feature matrix : {combined.shape}  |  features: {fcols}')
print(f'Train/Test     : {len(Xr_train):,} / {len(Xr_test):,}')


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_split=5,
    min_samples_leaf=3, max_features='sqrt', n_jobs=-1, random_state=42)
rf_model.fit(Xr_train, yr_train)
yr_pred = rf_model.predict(Xr_test)

rf_rmse = float(np.sqrt(mean_squared_error(yr_test, yr_pred)))
rf_mae  = float(np.mean(np.abs(yr_test - yr_pred)))
rf_r2   = float(r2_score(yr_test, yr_pred))

print(f'RF Forecast — RMSE: {rf_rmse:.4f}  MAE: {rf_mae:.4f}  R²: {rf_r2:.4f}')


In [ ]:
# Feature importance plot
importances = sorted(zip(fcols, rf_model.feature_importances_), key=lambda x: x[1], reverse=True)
f_names, f_imps = zip(*importances)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(list(f_names)[::-1], list(f_imps)[::-1], color='#6c5ce7')
axes[0].set_title('RF Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance')

axes[1].scatter(yr_test[:500], yr_pred[:500], alpha=0.3, color='#00b894', s=15)
mn, mx = min(yr_test.min(), yr_pred.min()), max(yr_test.max(), yr_pred.max())
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_title(f'Actual vs Predicted (R²={rf_r2:.3f})', fontweight='bold')
axes[1].set_xlabel('Actual Units')
axes[1].set_ylabel('Predicted Units')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODELS_DIR / 'model5_rf_forecast.png', bbox_inches='tight')
plt.show()

with open(MODELS_DIR / 'rf_forecast_model.pkl', 'wb') as f:
    pickle.dump({'model': rf_model, 'feature_cols': fcols}, f, protocol=pickle.HIGHEST_PROTOCOL)
print('✔ rf_forecast_model.pkl saved')


---
## 8 · Training Summary & Model Inventory

In [ ]:
print('\n' + '='*60)
print('  TRAINING COMPLETE — MODEL INVENTORY')
print('='*60)
for fpath in sorted(MODELS_DIR.glob('*.pkl')):
    size_kb = fpath.stat().st_size / 1024
    print(f'  {fpath.name:<40s}  {size_kb:8.1f} KB')
print()

metrics_report = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'models': {
        'similarity': {'n_products': len(product_ids), 'n_features': tfidf_matrix.shape[1]},
        'harmful':    {'mean_safety': sample_r['safety_score'].mean()},
        'rf_forecast': {'rmse': rf_rmse, 'mae': rf_mae, 'r2': rf_r2},
    }
}
with open(MODELS_DIR / 'training_report.json', 'w') as f:
    json.dump(metrics_report, f, indent=2, default=str)
print('✔ training_report.json saved')
print()
print('All models trained successfully! ✅')
